This collection of examples serves as a valuable reference for utilizing the Jupyter Notebook in the context of mechanical design. It not only demonstrates how to perform calculations, but also delves into the crucial aspect of conducting research in this field. The code closely aligns with the content found in the Roloff & Matek textbook's 6th edition, referencing equation and expression numbers.  
    
You are currently viewing a version of the code that **does not** include comments on how to write the code but instead focuses solely on the technical aspects. The xxx_extended.ipynb files do provide a detailed explanation of how to use the code.   

---

# Assignment

In this assignment, we will design the drive system for a transport belt. The drive will be powered by a geared electromotor and a chain transmission. Here are the initial specifications:

- The output power of the geared motor is given as $P_1 = 3 \: kW$ at an output speed of $n_1 = 125 \:rpm$.
- The desired speed of the shaft driving the transport belt is approximately $n_2 \approx 50 \:rpm$.
- The distance between the output shaft of the geared motor and the shaft driving the transport belt should be approximately $a_0 \approx 1000 \: mm$, with an angle between the shaft-to-shaft line and the horizontal of $\delta \approx 40°$
- For this specific application of the transport belt, we will assume the following use conditions: average starting, full load with moderate jolts, 8 hours of daily use, and a well-lubricated chain.

Your task is to calculate the following:

  a. Determine the number of teeth, $z_1$ and $z_2$, required for the chainwheels.  
  b. Select a suitable roller chain according to DIN 8187.  
  c. Identify the appropriate lubrication to be used.  
  d. Calculate the forces $F_A$ acting on the drive shaft.  

---
 

In [ ]:
# import statements
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

import MechDesign.Helpers as HM

from MechDesign.Units.Units import m_, mm_, kg_, s_, N_, rpm_, W_, deg_
import MechDesign.Units.UnitMethods as UM

import MechDesign.RnM as RnM

## a. Gear Ratio
By utilizing the required speeds, we can calculate the gear ratio, denoted as $i$. In a first step we will estimate $i'$.  The final gear ratio will depend on the number of teeth of both chain wheels. 

In [ ]:
# creating the chain to be calculated
CH = RnM.Chain()     #--- since Chain is part of the RnM library we call it as a part of it ---

In [3]:
# setting the value of known parameters
CH.n_1=125 * rpm_    
CH.n_2=50 * rpm_
t=HM.EqPrint('n_1',CH.n_1)     

Eq(n_1, 125*rpm_)

In [4]:
# An initial estimate of the gear ratio is needed; the final gear ratio will depend on the number of teeth on both gear wheels.
iprime = CH.n_1/CH.n_2
t=HM.EqPrint('iprime',iprime)       

Eq(iprime, 2.5)

For this transmission, the number of teeth mentioned in section 17.2.2 come closest to standard roller chain sizes: $z_1=23$ for the small wheel (selected from the range $z=17...25$, which is favorable for small wheels), and $z_2=57$ for the large wheel (determined by multiplying $z_1=23$ by $i'$ ).  
Therefore, the actual gear ratio becomes $i = z_2/z_1$

In [5]:
CH.z_1 = 23
CH.z_2 = 57

CH.i = CH.E17_1B_GearRatioTeeth() 
t=HM.EqPrint('i',CH.i*mm_)

Eq(i, 2.478*mm_)

**The solutions to quesion a. is:**
- $z_1 = 23$ with this smaler chain wheel beeing the driving wheel
- $z_2 = 57$ with this bigger geer beeing the driven wheel
  
the gear ratio then becomes $i \approx  2.48$   

---

## b. Chain selection

The selection of the chain size is made using the power diagram (Table 17-3). 
However, since the operating conditions here deviate from the values in the diagram, 
the desired diagram power must be determined according to equation 17.7.

In [6]:
t = HM.EqPrint('P_D',CH.E17_7_DiagramPower())

Eq(P_D, _K_A*_P_1*_f_1/(_f_2*_f_3*_f_4*_f_5*_f_6))

In [7]:
t = HM.MyHelp(CH.E17_7_DiagramPower())

_K_A*_P_1*_f_1/(_f_2*_f_3*_f_4*_f_5*_f_6)

    f_5 : [] power factor on lifetime expectancy 
    f_3 : [] power factor on shackle shape {exp 17.7}
    f_4 : [] power factor on number of chain wheels {exp 17.7}
    K_A : [] working condtion factor {table 3-5}
    P_1 : [W] input power
    f_6 : [] power factor on working conditions {table 17-7}
    f_2 : [] power factor on shaft to shaft distance {table 17-6}
    f_1 : [] power factor on number of teeth {table 17-5}


- The inital power $P_1=3000$ W
- The operating factor $K_A\approx1.6$, corresponding to the occurring conditions and according to Table 3-5b
- Estimating the number of teeth, from Table 17-5, a factor of $f_1\approx 0.8$ is read for $z_1=23$.
- The shaft distance factor $f_2$ is initially set to 1 because the chain pitch is still unknown.
- $f_3=1$ as we assume an even number of connecting links.
- $f_4=1$ for  $n=2$;
- $f_5=1$ for an assumed service life $L_h=15000$ hours.
- $f_6\approx 0.9$ according to Table 17.7, as a dust-free environment cannot be guaranteed.
 
With these values, the diagram power is calculated as follows

In [8]:
CH.P_1 = 3000*W_
CH.K_A = 1.6     #The operating factor K_A , corresponding to the occurring conditions and according to Table 3-5b
CH.f_1 = 0.8     #Estimating the number of teeth, from Table 17-5, a factor of f_1 = 0.8 is read for z_1=23.
CH.f_2 = 1       #The shaft distance factor f_2 is initially set to 1 because the chain pitch is still unknown.
CH.f_3 = 1       #as we assume an even number of connecting links.
CH.f_4 = 1       #for n = 2
CH.f_5 = 1       #for an, assumed service life of 15000hours
CH.f_6 = 0.9     #according to table 17.7 a dust free environment cannot be guaranteed
#---We will no calcuate the value P_D ---
CH.P_D = CH.E17_7_DiagramPower()
t=HM.EqPrint('P_D',CH.P_D)

Eq(P_D, 4267.0*W_)

For this power and for $n_1\approx125$ rpm, we select from Table 17-3:  
DIN 8187 - 16B - 1 : roller chain no. 16B, single strand. The pitch for this chain is $p=25.4$ mm.  
For the expected shaft distance $a_0\approx1000$ mm, the number of chain links can be calculated using equation 17.9.

In [9]:
CH.p = 25.4*mm_ 
CH.a_0 = 1000*mm_
CH.X_0 = CH.E17_9_InitalNrOfShackels()
t=HM.EqPrint('X_0',CH.X_0)

t=HM.EqPrint('X_0',CH.X_0.evalf())

Eq(X_0, 7.341/pi**2 + 118.7)

Eq(X_0, 119.5)

Following the recommendations in paragraph 17.2.5, it is preferred to have an even number of links. Therefore, the number of links is set to $X=120$.  
The effective shaft distance, $a$, can be calculated using expression 17.10

In [10]:
CH.X = 120
CH.a = CH.E17_10_ShaftDistance()
t=HM.EqPrint('a',CH.a.evalf())       

Eq(a, 1007.0*mm_)

**The solution to question b. is:**

- DIN 8187 - 16B - 1x120, chain
- single chain
- 120 links
- shaft distance $a=1007$ mm  


---

## c. lubrication 

The choice of lubrication depends on both the speed $v$ and the chain size (refer to Table 17-8).  
We can determine the speed by considering the diameter $d$ of one of the sprockets, utilizing expression 17.3

In [11]:
# opting to calcuate the speed using the smallest sprocket, opting for expression 17.3B, thus eliminating the need to calculate tau
CH.z = CH.z_1     #since 3B can be used for both d_1 and d_2, z has to be set to z_1 or z_2
CH.d_1 = CH.E17_3B_PitchDiameter()  
HM.EqPrint('d_1',CH.d_1.evalf())
CH.v = CH.E17_14_h_Speed()  #---Helper expressions within or adjacent to an expression are indicated using '_h_' or '_hA_','_hB_','_hC_,..
t=HM.EqPrint('v',CH.v)

Eq(d_1, 186.5*mm_)

Eq(v, 1588.0*mm_*rpm_/sin(0.04348*pi))

In [12]:
#---By adding 'evalf()', you can convert values like 'pi' into their numerical equivalents ---
t=HM.EqPrint('v',CH.v.evalf())

Eq(v, 11660.0*mm_*rpm_)

In [13]:
#---Utilize the 'UM.All_to_SI' method to convert all units to SI---
temp = CH.v.evalf()
temp = UM.All_to_SI(temp)
t=HM.EqPrint('v',temp)

Eq(v, 1.221*m_/s_)

Table 17-8 suggests that at a velocity of $v=1.22 m/s$ and with a size of 16B, drip lubrication (2) could still be used. However, to ensure  reliability, a precautionary measure is taken by opting for an oil bath (3).

**The solution to question c. is:**

- An oil bath is selected as the appropriate lubrication method for the chosen roller chain.

---

## d. Force on the drive shaft

The load on the drive shaft $F_A$ is slightly higher at the upper wheel due to the inclined position of the chain drive, corresponding to the difference between Fsn and Fso2. For the upper wheel, in accordance with equation 17.19, we have $F_A \approx F_t \cdot K_A + 2 \cdot F_{sb}$ , as per expression 17.17.  
The tangential force can be calculated using expression 17.14.


In [14]:
CH.F_t = CH.E17_14A_TangentialForce()
CH.F_t = UM.All_to_SI(CH.F_t)
CH.F_t = UM.kgm_s2_to_N(CH.F_t)
t=HM.EqPrint('F_t',CH.F_t)

Eq(F_t, 2457.0*N_)

Equation 17.17 facilitates the determination of the support force acting on the upper sprocket, but it requires the configuration of several parameters before it can be applied

In [15]:
CH.q = 2.7 *kg_/m_       # based on tabel 17-1
CH.g = 9.81*m_/s_**2
# first let's calculate d_2
CH.z = CH.z_2            # set z in order to calculate d_2, using expression 17.3
CH.d_2 = CH.E17_3B_PitchDiameter()
CH.epsilon_0 = CH.E17_17_hB_epsilon_0()
CH.l_T = CH.a * sp.cos(CH.epsilon_0)
# HM.EqPrint('l_T',CH.l_T.evalf())
CH.delta = 40*deg_       # as given in the assignemt
CH.psi = CH.E17_17_hA_psi()
CH.Fprime_s = 5          # based on tabel 17-4
CH.F_sb = CH.E17_17_TopWheelSupportForce()
CH.F_sb = UM.All_to_SI(CH.F_sb)
CH.F_sb = UM.kgm_s2_to_N(CH.F_sb)
t=HM.EqPrint('F_sb',CH.F_sb)

Eq(F_sb, 146.1*N_)

Compared to the circumferential force $F_t$, the support tension $F_{sb}$ is relatively small and can be readily disregarded in this scenario.  
Taking into consideration the operating factor $K_A \approx 1.6$ (refer to part b), the shaft load on the upper sprocket is calculated as:

In [16]:
CH.F_Ab = CH.E17_19B_ShaftSupportForceTopChainWheel()
t=HM.EqPrint('F_Ab',CH.F_Ab)

Eq(F_Ab, 4224.0*N_)

**The solution to question d. is:**  
  
- The support force for the upper sprocket will be $F_{Ab} \approx 4225$ N  


---  